In [1]:
import importlib
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "analysis" / "src").exists() and (candidate / "data").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError(f"Could not locate project root from {Path.cwd().resolve()}")
DROPBOX_ROOT = Path("/Users/satoshi/Library/CloudStorage/Dropbox/HSA_NKPC_MCMC")
sys.path.append(str(PROJECT_ROOT / "analysis" / "src"))
sys.path.append(str(PROJECT_ROOT / "analysis" / "gibbs"))

DATA_DIR = PROJECT_ROOT / "data"
IDATA_ROOT = DROPBOX_ROOT / "results" / "idata"
TEX_ROOT_DIR = PROJECT_ROOT / "results" / "tex" / "table"
FIG_ROOT = PROJECT_ROOT / "results" / "fig"

# Change DROPBOX_ROOT above if your Dropbox location differs.
# IDATA files will be saved to and loaded from IDATA_ROOT.
from func_data_build import build_dataset
from func_gibbs.gibbs_notebook_utils import display_hmc_results, display_hmc_posterior_prior, save_idata_map
from func_gibbs.gibbs_wrappers import draws_to_idata
import func_gibbs.gibbs_wrappers as gibbs_module
from func_gibbs.gibbs_wrappers import run_ces

data_dir = DATA_DIR
idat_dir = IDATA_ROOT
tex_root_dir = TEX_ROOT_DIR
base_fig_dir = FIG_ROOT

## 0. load data
data = build_dataset(data_dir)
data["DATE"] = pd.to_datetime(data.index)

## make model-specific maximum available sample
def make_model_sample(data, spec, inflation_specs, include_N=True):
    infl = inflation_specs[spec["inflation"]]
    gap = spec["gap"]

    required_cols = [
        infl["pi"],
        infl["pi_prev"],
        infl["pi_expect"],
        gap,
        f"{gap}_prev",
    ]

    if include_N:
        required_cols.append(spec["N"])

    missing_cols = [col for col in required_cols if col not in data.columns]
    if missing_cols:
        raise KeyError(f"Missing columns for {spec['model_id']}: {missing_cols}")

    sample = (
        data[required_cols]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .copy()
    )

    if sample.empty:
        raise ValueError(f"No valid sample for {spec['model_id']}")

    return sample

In [2]:
## model specifications
inflation_specs = {
    "ppi": {
        "pi": "pi_ppi",
        "pi_prev": "pi_ppi_prev",
        "pi_expect": "Epi_spf_gdp",
    },
    "cpi": {
        "pi": "pi_cpi",
        "pi_prev": "pi_cpi_prev",
        "pi_expect": "Epi_spf_cpi",
    },
}

model_grid = [
    {
        "model_id": "S1_U_G",
        "set": 1,
        "inflation": "ppi",
        "gap": "unemp_gap",
        "N": "N_Gustavo",
    },
    {
        "model_id": "S1_U_T",
        "set": 1,
        "inflation": "ppi",
        "gap": "unemp_gap",
        "N": "N_TNIC",
    },
    {
        "model_id": "S1_Y_G",
        "set": 1,
        "inflation": "ppi",
        "gap": "output_gap_BN",
        "N": "N_Gustavo",
    },
    {
        "model_id": "S1_Y_T",
        "set": 1,
        "inflation": "ppi",
        "gap": "output_gap_BN",
        "N": "N_TNIC",
    },
    {
        "model_id": "S2_U_G",
        "set": 2,
        "inflation": "cpi",
        "gap": "unemp_gap",
        "N": "N_Gustavo",
    },
    {
        "model_id": "S2_U_T",
        "set": 2,
        "inflation": "cpi",
        "gap": "unemp_gap",
        "N": "N_TNIC",
    },
    {
        "model_id": "S2_Y_G",
        "set": 2,
        "inflation": "cpi",
        "gap": "output_gap_BN",
        "N": "N_Gustavo",
    },
    {
        "model_id": "S2_Y_T",
        "set": 2,
        "inflation": "cpi",
        "gap": "output_gap_BN",
        "N": "N_TNIC",
    },
]

## 4. check sample periods

sample_info = []

for spec in model_grid:
    sample = make_model_sample(
        data=data,
        spec=spec,
        inflation_specs=inflation_specs,
        include_N=True,
    )

    sample_info.append({
        "model_id": spec["model_id"],
        "set": spec["set"],
        "inflation": spec["inflation"],
        "gap": spec["gap"],
        "N": spec["N"],
        "start": sample.index.min(),
        "end": sample.index.max(),
        "T": len(sample),
    })

sample_info_df = pd.DataFrame(sample_info)
display(sample_info_df)

,model_id,set,inflation,gap,N,start,end,T
0,S1_U_G,1,ppi,unemp_gap,N_Gustavo,1974-03-31 23:59:59.999999999,2012-12-31 23:59:59.999999999,155
1,S1_U_T,1,ppi,unemp_gap,N_TNIC,1988-03-31 23:59:59.999999999,2022-12-31 23:59:59.999999999,140
2,S1_Y_G,1,ppi,output_gap_BN,N_Gustavo,1974-03-31 23:59:59.999999999,2012-12-31 23:59:59.999999999,155
3,S1_Y_T,1,ppi,output_gap_BN,N_TNIC,1988-03-31 23:59:59.999999999,2022-12-31 23:59:59.999999999,140
4,S2_U_G,2,cpi,unemp_gap,N_Gustavo,1981-09-30 23:59:59.999999999,2012-12-31 23:59:59.999999999,126
5,S2_U_T,2,cpi,unemp_gap,N_TNIC,1988-03-31 23:59:59.999999999,2022-12-31 23:59:59.999999999,140
6,S2_Y_G,2,cpi,output_gap_BN,N_Gustavo,1981-09-30 23:59:59.999999999,2012-12-31 23:59:59.999999999,126
7,S2_Y_T,2,cpi,output_gap_BN,N_TNIC,1988-03-31 23:59:59.999999999,2022-12-31 23:59:59.999999999,140


In [3]:
## prior distributions and MCMC parameters
seed = 0
n_iter = 12000
burn = 4000
thin = 5

prior_specs = {
    "alpha": (0.5, 1),
    "kappa": (0.1, 1),
    "phi_1": (0.7, 1),
    "rho": (0.0, 1),
}

display(pd.DataFrame([
    {"parameter": key, "prior_mean": value[0], "prior_sd": value[1]}
    for key, value in prior_specs.items()
]))


## 6. Gibbs MCMC: CES models

gibbs_module = importlib.reload(gibbs_module)
run_ces = gibbs_module.run_ces
draws_to_idata = gibbs_module.draws_to_idata

dict_draws = {}
dict_idata = {}

for spec in model_grid:
    # include_N=True keeps the CES sample aligned with the corresponding HSA sample.
    # If you want CES to use the longest possible sample independent of N, set include_N=False.
    sample = make_model_sample(
        data=data,
        spec=spec,
        inflation_specs=inflation_specs,
        include_N=True,
    )

    infl = inflation_specs[spec["inflation"]]
    gap = spec["gap"]

    pi = np.asarray(sample[infl["pi"]], dtype=float)
    pi_prev = np.asarray(sample[infl["pi_prev"]], dtype=float)
    pi_expect = np.asarray(sample[infl["pi_expect"]], dtype=float)

    x = np.asarray(sample[gap], dtype=float)
    x_prev = np.asarray(sample[f"{gap}_prev"], dtype=float)

    for orth in (False, True):
        label = "corr" if not orth else "uncorr"
        model_name = f"{spec['model_id']}_CES_{label}"

        print(
            "Running",
            model_name,
            "| sample:",
            sample.index.min(),
            "to",
            sample.index.max(),
            "| T =",
            len(sample),
        )

        draws = run_ces(
            pi=pi,
            pi_prev=pi_prev,
            pi_expect=pi_expect,
            x=x,
            x_prev=x_prev,
            prior_specs=prior_specs,
            n_iter=n_iter,
            burn=burn,
            thin=thin,
            rng=np.random.default_rng(seed),
            orth=orth,
        )

        dict_draws[model_name] = draws
        dict_idata[model_name] = draws_to_idata(draws)

list(dict_idata.keys())

save_idata_map(dict_idata, idat_dir)
print(f"Saved {len(dict_idata)} idata files to {idat_dir}")



,parameter,prior_mean,prior_sd
0,alpha,0.5,1
1,kappa,0.1,1
2,phi_1,0.7,1
3,rho,0.0,1


Running S1_U_G_CES_corr | sample: 1974-03-31 23:59:59.999999999 to 2012-12-31 23:59:59.999999999 | T = 155
Running S1_U_G_CES_uncorr | sample: 1974-03-31 23:59:59.999999999 to 2012-12-31 23:59:59.999999999 | T = 155
Running S1_U_T_CES_corr | sample: 1988-03-31 23:59:59.999999999 to 2022-12-31 23:59:59.999999999 | T = 140
Running S1_U_T_CES_uncorr | sample: 1988-03-31 23:59:59.999999999 to 2022-12-31 23:59:59.999999999 | T = 140
Running S1_Y_G_CES_corr | sample: 1974-03-31 23:59:59.999999999 to 2012-12-31 23:59:59.999999999 | T = 155
Running S1_Y_G_CES_uncorr | sample: 1974-03-31 23:59:59.999999999 to 2012-12-31 23:59:59.999999999 | T = 155
Running S1_Y_T_CES_corr | sample: 1988-03-31 23:59:59.999999999 to 2022-12-31 23:59:59.999999999 | T = 140
Running S1_Y_T_CES_uncorr | sample: 1988-03-31 23:59:59.999999999 to 2022-12-31 23:59:59.999999999 | T = 140
Running S2_U_G_CES_corr | sample: 1981-09-30 23:59:59.999999999 to 2012-12-31 23:59:59.999999999 | T = 126
Running S2_U_G_CES_uncorr | s

In [4]:
## posterior summary
display_hmc_results(
    dict_idata,
    prior_specs,
    models_to_show=list(dict_idata.keys()),
    params=("alpha", "kappa", "phi_1", "rho", "sigma_e", "sigma_zeta"),
    title="Posterior summary for CES models",
)

,model,alpha,kappa,phi_1,rho,sigma_e,sigma_zeta
0,S1_U_G_CES_corr,0.8220,0.1049,0.9357,0.1174,2.5206,0.6512
1,S1_U_G_CES_uncorr,0.8132,0.1611,0.9387,0.0000,2.5056,0.6517
2,S1_U_T_CES_corr,0.8548,-0.2554,0.8489,0.2770,3.0164,0.9726
3,S1_U_T_CES_uncorr,0.8564,-0.0228,0.8527,0.0000,2.9754,0.9736
4,S1_Y_G_CES_corr,0.8474,0.2685,0.8960,0.0467,2.4940,0.7053
5,S1_Y_G_CES_uncorr,0.8416,0.2900,0.8999,0.0000,2.4777,0.7058
6,S1_Y_T_CES_corr,0.8526,0.2603,0.7284,0.2217,2.9145,0.9754
7,S1_Y_T_CES_uncorr,0.8334,0.5753,0.7349,0.0000,2.8611,0.9765
8,S2_U_G_CES_corr,0.6831,0.0683,0.9541,0.0632,0.6930,0.5942
9,S2_U_G_CES_uncorr,0.6759,0.0762,0.9573,0.0000,0.6870,0.5942


Posterior summary for CES models
                model       param      mean    ci_2.5   ci_97.5
0     S1_U_G_CES_corr       alpha  0.821968  0.737556  0.906777
1     S1_U_G_CES_corr       kappa  0.104853 -0.114610  0.327024
2     S1_U_G_CES_corr       phi_1  0.935681  0.880579  0.993945
3     S1_U_G_CES_corr         rho  0.117428 -0.045358  0.272850
4     S1_U_G_CES_corr     sigma_e  2.520600  2.254943  2.818376
..                ...         ...       ...       ...       ...
91  S2_Y_T_CES_uncorr       kappa  0.172659  0.090641  0.250636
92  S2_Y_T_CES_uncorr       phi_1  0.734907  0.621806  0.852354
93  S2_Y_T_CES_uncorr         rho  0.000000  0.000000  0.000000
94  S2_Y_T_CES_uncorr     sigma_e  0.696313  0.621023  0.785329
95  S2_Y_T_CES_uncorr  sigma_zeta  0.976452  0.869591  1.099393

[96 rows x 5 columns]
